In [ ]:
!pip install pandas openpyxl tqdm
# =====================================================
# Step-2.1 Charged Data Cleaning no items limit
# =====================================================

import pandas as pd, ast
from tqdm import tqdm
from google.colab import files
from datetime import datetime

print("📁 Upload file")
uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_excel(file)

clean_rows = []
max_items = 0  # 🔥 track max items for column creation

for _, row in tqdm(df.iterrows(), total=len(df)):

    try:
        # =========================
        # TS FIX
        # =========================
        ts_val = str(row.get("ts", ""))
        try:
            ts_val = datetime.strptime(ts_val, "%Y%m%d%H%M%S").strftime("%Y-%m-%d")
        except:
            ts_val = ""

        # =========================
        # BASE
        # =========================
        base = {
            "identity": row.get("profile.identity", ""),
            "invoice_number": row.get("event_props.Invoice Number", ""),
            "type": "event",
            "event_name": "Charged",
            "api_type": "CleverTapApi",
            "ts": ts_val,

            "Tender Name": row.get("event_props.Tender Name", ""),
            "CUID Number": row.get("event_props.CUID Number", ""),
            "Invoice Number": row.get("event_props.Invoice Number", ""),
            "Invoice Type": row.get("event_props.Invoice Type", ""),
            "Store Code": row.get("event_props.Store Code", ""),
            "Store City": row.get("event_props.Store City", ""),
            "Billing Store Zone": row.get("event_props.Billing Store Zone", ""),
            "Total Billing Value": row.get("event_props.Total Billing Value", ""),
            "Total MRP Value": row.get("event_props.Total MRP Value", "")
        }

        # =========================
        # ITEMS (NO LIMIT)
        # =========================
        items_raw = row.get("items", "[]")

        try:
            items = ast.literal_eval(items_raw)
        except:
            items = []

        if not isinstance(items, list):
            items = []

        max_items = max(max_items, len(items))  # track max

        for i, item in enumerate(items, start=1):
            base[f"Item{i}_Billing Discount Percentage"] = item.get("Items|Billing Discount Percentage", "")
            base[f"Item{i}_Billing Value"] = item.get("Items|Billing Value", "")
            base[f"Item{i}_Invoice Brand"] = item.get("Items|Invoice Brand", "")
            base[f"Item{i}_MRP"] = item.get("Items|MRP", "")
            base[f"Item{i}_Model Number"] = item.get("Items|Model Number", "")

        # =========================
        # IDENTITIES (LAST)
        # =========================
        identities = row.get("profile.all_identities", "[]")
        try:
            identities = ast.literal_eval(identities)
        except:
            identities = []

        for i, val in enumerate(identities, start=1):
            base[f"identity_{i}"] = val

        clean_rows.append(base)

    except:
        pass

# =========================
# DATAFRAME
# =========================
clean_df = pd.DataFrame(clean_rows)

# =========================
# COLUMN ORDER (DYNAMIC ITEMS)
# =========================
first_cols = [
    "identity","invoice_number","type","event_name","api_type","ts",
    "Tender Name","CUID Number","Invoice Number","Invoice Type",
    "Store Code","Store City","Billing Store Zone",
    "Total Billing Value","Total MRP Value"
]

# 🔥 dynamic item columns
item_cols = []
for i in range(1, max_items + 1):
    item_cols += [
        f"Item{i}_Billing Discount Percentage",
        f"Item{i}_Billing Value",
        f"Item{i}_Invoice Brand",
        f"Item{i}_MRP",
        f"Item{i}_Model Number"
    ]

# identities at last
identity_cols = [c for c in clean_df.columns if c.startswith("identity_")]

final_cols = first_cols + item_cols + identity_cols
final_cols = [c for c in final_cols if c in clean_df.columns]

clean_df = clean_df[final_cols]

# =========================
# SAVE
# =========================
output = "Final_Output_No_Item_Limit.xlsx"
clean_df.to_excel(output, index=False)

print(f"✅ Done: {len(clean_df)} rows | Max Items: {max_items}")
files.download(output)